# Hospital Patient Care Analytics Pipeline

**Calibo AI/ML Learning Assignment**

This project demonstrates an end-to-end ETL and machine-learning workflow for the hospital patient-care analytics case study.

> **Dataset note:** The supplied dataset is synthetic and is used only for educational demonstration. The `High_Risk` target is artificially generated and is not a clinically validated label.

## 1. ETL Architecture

**Extract → Transform → Validate → Load → Analyze → Predict**

- **Extract:** Read hospital patient data from CSV.
- **Transform:** Remove duplicates, clean data, handle missing values and create an operational feature.
- **Validate:** Check schema, missing values and target values.
- **Load:** Save the cleaned dataset.
- **Analyze:** Perform EDA and waiting-time analysis.
- **Predict:** Train classification models for potentially high-risk patients.

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

from extract import extract_data
from transform import transform_data
from validate import validate_data
from load import load_data
from config import RISK_FEATURES, TARGET_COLUMN

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 2. Extract

In [ ]:
df_raw = extract_data()
df_raw.head()

## 3. Transform and Validate

In [ ]:
df = transform_data(df_raw)
validate_data(df)

## 4. Load

In [ ]:
output_path = load_data(df)
print(output_path)

## 5. Data Understanding

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.describe(include="all").T)

## 6. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["Age"], bins=20, kde=True)
plt.title("Age Distribution of Patients")
plt.xlabel("Age")
plt.ylabel("Number of Patients")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Department")
plt.title("Patients by Department")
plt.xticks(rotation=30)
plt.show()

In [ ]:
department_wait = df.groupby("Department")["Waiting_Time_Min"].mean().sort_values(ascending=False)
display(department_wait.to_frame("Average_Waiting_Time_Min"))

plt.figure(figsize=(9, 5))
department_wait.plot(kind="bar")
plt.title("Average Waiting Time by Department")
plt.ylabel("Minutes")
plt.xticks(rotation=30)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COLUMN)
plt.title("High-Risk vs Low-Risk Patients")
plt.xlabel("High Risk (0 = Low, 1 = High)")
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

## 7. Machine Learning

In [ ]:
X = df[RISK_FEATURES]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

### 7.1 Logistic Regression

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)

log_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred_log),
    "Precision": precision_score(y_test, y_pred_log),
    "Recall": recall_score(y_test, y_pred_log),
    "F1 Score": f1_score(y_test, y_pred_log),
}
print(classification_report(y_test, y_pred_log))
print(log_metrics)

In [ ]:
cm = confusion_matrix(y_test, y_pred_log)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### 7.2 Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf),
    "Recall": recall_score(y_test, y_pred_rf),
    "F1 Score": f1_score(y_test, y_pred_rf),
}
print(classification_report(y_test, y_pred_rf))
print(rf_metrics)

## 8. Model Comparison

In [ ]:
results = pd.DataFrame([log_metrics, rf_metrics])
display(results)

## 9. Feature Importance

In [ ]:
importance = pd.DataFrame({
    "Feature": RISK_FEATURES,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Feature Importance - Random Forest")
plt.show()

## 10. Example Patient Prediction

In [ ]:
new_patient = pd.DataFrame({
    "Age": [68],
    "Heart_Rate": [105],
    "Systolic_BP": [165],
    "Diastolic_BP": [95],
    "SpO2": [91],
    "Glucose": [185],
    "Previous_Admissions": [3],
    "Previous_Visits": [8],
    "Abnormal_Lab_Count": [4]
})

prediction = rf_model.predict(new_patient)[0]
probability = rf_model.predict_proba(new_patient)[0]

print("Predicted class:", "High Risk" if prediction == 1 else "Low Risk")
print(f"Low-risk probability : {probability[0]:.2%}")
print(f"High-risk probability: {probability[1]:.2%}")

## 11. Business Insights

- Department-wise waiting time can help identify operational bottlenecks.
- Patient demographic and clinical variables can be analyzed for risk patterns.
- ML classification can flag potentially high-risk patients for additional professional review.
- Feature importance provides an interpretable view of which variables influenced the model.
- The pipeline demonstrates how raw hospital data can be converted into analytics-ready data and ML predictions.

## 12. Conclusion

The project implements a complete ETL and AI/ML workflow: data extraction, transformation, validation, loading, exploratory analysis, model training, evaluation and example prediction.

**Important:** This is an educational project using synthetic data. Model predictions are not medical diagnoses and should not be used for clinical decisions.